# Bab 13: Data Sekuensial

Notebook pendamping buku *Pengenalan Pola dan Machine Learning: Pendekatan Probabilistik dengan Python*.

Jalankan sel dari atas ke bawah (**Runtime → Run all** di Colab). Sel pertama menyiapkan folder kode dan modul data sintetis; setiap sel berikutnya adalah satu listing dari buku, dengan nomor penanda `# (1)`, `# (2)`, … yang dijelaskan di bawah listing pada buku. Angka keluaran bisa sedikit berbeda dari buku bila versi pustakanya berbeda (lihat Lampiran A).

In [ ]:
# Menyiapkan folder kode/ dan modul pendukung bab ini
import os, pathlib
BERKAS = {
"bab13/data_pos.py": "\"\"\"Korpus kecil kalimat bahasa Indonesia berlabel kelas kata (sintetis).\n\nKalimat dibangkitkan dari pola kelas kata dan leksikon sederhana. Beberapa kata\nsengaja taksa (ambigu), misalnya \"tahu\" (NOUN: makanan, VERB: mengetahui),\n\"bisa\" (MD: dapat, NOUN: racun), \"jalan\", \"masak\", dan \"baru\".\n\"\"\"\nimport numpy as np\n\nTAG = [\"PRON\", \"NOUN\", \"VERB\", \"ADJ\", \"ADP\", \"ADV\", \"MD\", \"DET\"]\nLEKSIKON = {\n    \"PRON\": [\"saya\", \"dia\", \"kami\", \"mereka\", \"kita\", \"aku\", \"kamu\", \"beliau\"],\n    \"NOUN\": [\"nasi\", \"tahu\", \"tempe\", \"pasar\", \"sekolah\", \"rumah\", \"jalan\", \"buku\",\n             \"kopi\", \"sawah\", \"bisa\", \"ular\", \"sore\", \"pagi\", \"kantor\", \"kereta\"],\n    \"VERB\": [\"makan\", \"masak\", \"membeli\", \"pergi\", \"membaca\", \"tahu\", \"jalan\",\n             \"minum\", \"menjual\", \"datang\", \"pulang\", \"melihat\"],\n    \"ADJ\":  [\"enak\", \"panas\", \"baru\", \"besar\", \"murah\", \"masak\", \"ramai\", \"jauh\"],\n    \"ADP\":  [\"di\", \"ke\", \"dari\", \"untuk\", \"dengan\"],\n    \"ADV\":  [\"baru\", \"sudah\", \"sedang\", \"sering\", \"tidak\", \"juga\"],\n    \"MD\":   [\"bisa\", \"akan\", \"harus\", \"mau\", \"boleh\"],\n    \"DET\":  [\"itu\", \"ini\"],\n}\nPOLA = [\n    \"PRON VERB NOUN\", \"PRON VERB NOUN ADJ\", \"PRON ADV VERB NOUN\",\n    \"PRON MD VERB NOUN ADP NOUN\", \"NOUN DET ADJ\", \"NOUN DET ADV ADJ\",\n    \"PRON VERB ADP NOUN\", \"PRON ADV VERB ADP NOUN NOUN\", \"NOUN NOUN DET ADJ\",\n    \"PRON MD VERB ADP NOUN\", \"PRON VERB NOUN DET\", \"NOUN ADJ DET ADV ADJ\",\n    \"PRON ADV MD VERB NOUN\", \"PRON VERB NOUN ADP NOUN DET\",\n]\n\ndef buat_korpus(n=3000, seed=0):\n    rng = np.random.default_rng(seed)\n    p_pola = rng.dirichlet(np.ones(len(POLA)) * 2)\n    korpus = []\n    for _ in range(n):\n        tags = POLA[rng.choice(len(POLA), p=p_pola)].split()\n        kata = [LEKSIKON[t][min(int(rng.exponential(len(LEKSIKON[t]) / 3)),\n                                len(LEKSIKON[t]) - 1)] for t in tags]  # Zipf kasar\n        korpus.append(list(zip(kata, tags)))\n    return korpus\n",
"bab13/hmm_pos.py": "import numpy as np\nfrom collections import Counter\nfrom data_pos import buat_korpus, TAG\n\nkorpus = buat_korpus()\nlatih, uji = korpus[:2500], korpus[2500:]\nkosakata = sorted({w for s in korpus for w, _ in s}); iw = {w: i for i, w in enumerate(kosakata)}\nK, V = len(TAG), len(kosakata); it = {t: i for i, t in enumerate(TAG)}\n\npi, A, B = np.ones(K), np.ones((K, K)), np.ones((K, V))   # penghalusan +1  #<1>\nfor s in latih:\n    t = [it[g] for _, g in s]; pi[t[0]] += 1\n    for a, b in zip(t[:-1], t[1:]): A[a, b] += 1\n    for (w, _), k in zip(s, t): B[k, iw[w]] += 1\npi, A, B = pi / pi.sum(), A / A.sum(1, keepdims=True), B / B.sum(1, keepdims=True)\nlpi, lA, lB = np.log(pi), np.log(A), np.log(B)\n\ndef viterbi(kata):                                         #<2>\n    o = [iw[w] for w in kata]; d = lpi + lB[:, o[0]]; asal = []\n    for x in o[1:]:\n        c = d[:, None] + lA; asal.append(c.argmax(0)); d = c.max(0) + lB[:, x]\n    jalur = [int(d.argmax())]\n    for a in reversed(asal): jalur.append(int(a[jalur[-1]]))\n    return [TAG[k] for k in jalur[::-1]]\n\ndef maju_mundur(kata):                                     #<3>\n    o = [iw[w] for w in kata]; T = len(o)\n    al, be = np.zeros((T, K)), np.ones((T, K))\n    al[0] = pi * B[:, o[0]]\n    for n in range(1, T): al[n] = (al[n-1] @ A) * B[:, o[n]]\n    for n in range(T - 2, -1, -1): be[n] = A @ (B[:, o[n+1]] * be[n+1])\n    return al * be / al[-1].sum(), al[-1].sum()\n\nterbanyak = {}                                             # pembanding  #<4>\nfor w in kosakata:\n    c = Counter(g for s in latih for x, g in s if x == w)\n    terbanyak[w] = c.most_common(1)[0][0]\ntaksa = {w for w in kosakata if len({g for s in korpus for x, g in s if x == w}) > 1}\nbenar = {\"hmm\": [0, 0], \"base\": [0, 0]}\nfor s in uji:\n    kata, gold = zip(*s); pred = viterbi(kata)\n    for w, g, p in zip(kata, gold, pred):\n        j = int(w in taksa)\n        benar[\"hmm\"][j] += p == g; benar[\"base\"][j] += terbanyak[w] == g\nn_taksa = sum(w in taksa for s in uji for w, _ in s); n = sum(len(s) for s in uji)\nprint(\"kata taksa:\", sorted(taksa))\nprint(f\"akurasi tag terbanyak : semua {(sum(benar['base']))/n:.3f},\"\n      f\" kata taksa {benar['base'][1]/n_taksa:.3f}\")\nprint(f\"akurasi HMM + Viterbi : semua {(sum(benar['hmm']))/n:.3f},\"\n      f\" kata taksa {benar['hmm'][1]/n_taksa:.3f}\")\nfor kal in [\"beliau baru masak tahu\", \"tahu itu baru masak\", \"saya tahu jalan itu\"]:\n    print(f\"  {kal:24s} -> {' '.join(viterbi(kal.split()))}\")\ng, _ = maju_mundur(\"tahu itu baru masak\".split())\nprint(\"p(tag | kalimat) untuk 'masak':\",\n      \", \".join(f\"{t} {g[3, i]:.2f}\" for i, t in enumerate(TAG) if g[3, i] > 0.01))\nfor kal in [\"saya makan nasi enak\", \"nasi enak saya makan\"]:   # peluang kalimat\n    print(f\"ln p('{kal}') = {np.log(maju_mundur(kal.split())[1]):.2f}\")\n",
"bab13/kalman_kapal.py": "import numpy as np\n\n# Kapal di selat (ilustrasi): keadaan z = [x, y, vx, vy] dalam km dan km/menit.\nrng = np.random.default_rng(2)\nT, dt = 120, 1.0\nA = np.eye(4); A[0, 2] = A[1, 3] = dt                   # kecepatan konstan  #<1>\nG = np.eye(4)[:2]                                       # hanya posisi teramati\nQ = np.diag([0, 0, 1e-4, 1e-4])                         # percepatan acak kecil\nR = 0.4 ** 2 * np.eye(2)                                # derau posisi 0,4 km\nz = np.zeros((T, 4)); z[0] = [0, 0, 0.30, 0.05]\nfor n in range(1, T):\n    z[n] = A @ z[n-1] + rng.multivariate_normal(np.zeros(4), Q)\n    if n == 78: z[n, 2:] = [0.10, 0.28]                 # berbelok saat sinyal hilang\nx = z[:, :2] + rng.normal(0, 0.4, (T, 2))\nada = np.ones(T, bool); ada[70:90] = False              # sinyal hilang 20 menit\n\ndef kalman(x, ada):                                     #<2>\n    m, P = np.r_[x[0], 0, 0], np.diag([0.16, 0.16, 1, 1])\n    M, S, Mp, Sp = [], [], [], []\n    for n in range(T):\n        if n > 0:                                       # prediksi\n            m, P = A @ m, A @ P @ A.T + Q\n        Mp.append(m); Sp.append(P)\n        if ada[n]:                                      # koreksi dengan pengamatan\n            Kg = P @ G.T @ np.linalg.inv(G @ P @ G.T + R)\n            m, P = m + Kg @ (x[n] - G @ m), (np.eye(4) - Kg @ G) @ P\n        M.append(m); S.append(P)\n    return map(np.array, (M, S, Mp, Sp))\n\ndef rts(M, S, Mp, Sp):                                  # penghalus  #<3>\n    Mh, Sh = M.copy(), S.copy()\n    for n in range(T - 2, -1, -1):\n        J = S[n] @ A.T @ np.linalg.inv(Sp[n+1])\n        Mh[n] = M[n] + J @ (Mh[n+1] - Mp[n+1])\n        Sh[n] = S[n] + J @ (Sh[n+1] - Sp[n+1]) @ J.T\n    return Mh, Sh\n\nM, S, Mp, Sp = kalman(x, ada); Mh, Sh = rts(M, S, Mp, Sp)\ngalat = lambda e, m: np.sqrt(np.mean(np.sum((e[m, :2] - z[m, :2]) ** 2, 1)))\nfor nama, e in [(\"pengamatan\", np.c_[x, x]), (\"filter\", M), (\"penghalus\", Mh)]:\n    print(f\"RMSE {nama:10s}: saat ada sinyal {galat(e, ada):.3f} km\"\n          + (\"\" if nama == \"pengamatan\" else f\", saat hilang {galat(e, ~ada):.3f} km\"))\nsd = lambda S_, n: np.sqrt(S_[n, 0, 0])\nprint(f\"simp. baku posisi-x (filter): menit 69 {sd(S, 69):.2f}, menit 89 {sd(S, 89):.2f},\"\n      f\" menit 90 {sd(S, 90):.2f}\")\n",
"bab13/rantai_markov.py": "import numpy as np\n\n# Barisan harian 0 = kering, 1 = hujan (sintetis, ilustrasi musim peralihan).\nrng = np.random.default_rng(0)\nA_benar = np.array([[0.8, 0.2], [0.4, 0.6]])          # p(besok | hari ini)\nx = [0]\nfor _ in range(729):\n    x.append(rng.choice(2, p=A_benar[x[-1]]))\nx = np.array(x)\n\nC = np.zeros((2, 2))                                   # hitung transisi  #<1>\nnp.add.at(C, (x[:-1], x[1:]), 1)\nA = C / C.sum(1, keepdims=True)\nprint(\"matriks transisi taksiran:\\n\", A.round(3))\nw, V = np.linalg.eig(A.T)                              # distribusi stasioner  #<2>\npi = np.real(V[:, np.argmax(np.real(w))]); pi /= pi.sum()\nprint(\"distribusi stasioner:\", pi.round(3), \"| porsi hujan di data:\", x.mean().round(3))\nfor n in [1, 2, 7]:                                    # peluang n hari ke depan  #<3>\n    print(f\"p(hujan {n} hari lagi | hari ini hujan) = {np.linalg.matrix_power(A, n)[1, 1]:.3f}\")\n"
}
akar = pathlib.Path('pola-ml/kode').resolve()
for nama, isi in BERKAS.items():
    p = akar / nama; p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(isi, encoding='utf-8')
os.chdir(akar / 'bab13')
print('folder kerja:', os.getcwd())

## Listing 13.1: Rantai Markov hujan harian

Berkas: `kode/bab13/rantai_markov.py`

In [ ]:
import numpy as np

# Barisan harian 0 = kering, 1 = hujan (sintetis, ilustrasi musim peralihan).
rng = np.random.default_rng(0)
A_benar = np.array([[0.8, 0.2], [0.4, 0.6]])          # p(besok | hari ini)
x = [0]
for _ in range(729):
    x.append(rng.choice(2, p=A_benar[x[-1]]))
x = np.array(x)

C = np.zeros((2, 2))                                   # hitung transisi  # (1)
np.add.at(C, (x[:-1], x[1:]), 1)
A = C / C.sum(1, keepdims=True)
print("matriks transisi taksiran:\n", A.round(3))
w, V = np.linalg.eig(A.T)                              # distribusi stasioner  # (2)
pi = np.real(V[:, np.argmax(np.real(w))]); pi /= pi.sum()
print("distribusi stasioner:", pi.round(3), "| porsi hujan di data:", x.mean().round(3))
for n in [1, 2, 7]:                                    # peluang n hari ke depan  # (3)
    print(f"p(hujan {n} hari lagi | hari ini hujan) = {np.linalg.matrix_power(A, n)[1, 1]:.3f}")

## Listing 13.2: HMM untuk penandaan kelas kata: pelatihan, Viterbi, dan maju–mundur

Berkas: `kode/bab13/hmm_pos.py`

In [ ]:
import numpy as np
from collections import Counter
from data_pos import buat_korpus, TAG

korpus = buat_korpus()
latih, uji = korpus[:2500], korpus[2500:]
kosakata = sorted({w for s in korpus for w, _ in s}); iw = {w: i for i, w in enumerate(kosakata)}
K, V = len(TAG), len(kosakata); it = {t: i for i, t in enumerate(TAG)}

pi, A, B = np.ones(K), np.ones((K, K)), np.ones((K, V))   # penghalusan +1  # (1)
for s in latih:
    t = [it[g] for _, g in s]; pi[t[0]] += 1
    for a, b in zip(t[:-1], t[1:]): A[a, b] += 1
    for (w, _), k in zip(s, t): B[k, iw[w]] += 1
pi, A, B = pi / pi.sum(), A / A.sum(1, keepdims=True), B / B.sum(1, keepdims=True)
lpi, lA, lB = np.log(pi), np.log(A), np.log(B)

def viterbi(kata):                                         # (2)
    o = [iw[w] for w in kata]; d = lpi + lB[:, o[0]]; asal = []
    for x in o[1:]:
        c = d[:, None] + lA; asal.append(c.argmax(0)); d = c.max(0) + lB[:, x]
    jalur = [int(d.argmax())]
    for a in reversed(asal): jalur.append(int(a[jalur[-1]]))
    return [TAG[k] for k in jalur[::-1]]

def maju_mundur(kata):                                     # (3)
    o = [iw[w] for w in kata]; T = len(o)
    al, be = np.zeros((T, K)), np.ones((T, K))
    al[0] = pi * B[:, o[0]]
    for n in range(1, T): al[n] = (al[n-1] @ A) * B[:, o[n]]
    for n in range(T - 2, -1, -1): be[n] = A @ (B[:, o[n+1]] * be[n+1])
    return al * be / al[-1].sum(), al[-1].sum()

terbanyak = {}                                             # pembanding  # (4)
for w in kosakata:
    c = Counter(g for s in latih for x, g in s if x == w)
    terbanyak[w] = c.most_common(1)[0][0]
taksa = {w for w in kosakata if len({g for s in korpus for x, g in s if x == w}) > 1}
benar = {"hmm": [0, 0], "base": [0, 0]}
for s in uji:
    kata, gold = zip(*s); pred = viterbi(kata)
    for w, g, p in zip(kata, gold, pred):
        j = int(w in taksa)
        benar["hmm"][j] += p == g; benar["base"][j] += terbanyak[w] == g
n_taksa = sum(w in taksa for s in uji for w, _ in s); n = sum(len(s) for s in uji)
print("kata taksa:", sorted(taksa))
print(f"akurasi tag terbanyak : semua {(sum(benar['base']))/n:.3f},"
      f" kata taksa {benar['base'][1]/n_taksa:.3f}")
print(f"akurasi HMM + Viterbi : semua {(sum(benar['hmm']))/n:.3f},"
      f" kata taksa {benar['hmm'][1]/n_taksa:.3f}")
for kal in ["beliau baru masak tahu", "tahu itu baru masak", "saya tahu jalan itu"]:
    print(f"  {kal:24s} -> {' '.join(viterbi(kal.split()))}")
g, _ = maju_mundur("tahu itu baru masak".split())
print("p(tag | kalimat) untuk 'masak':",
      ", ".join(f"{t} {g[3, i]:.2f}" for i, t in enumerate(TAG) if g[3, i] > 0.01))
for kal in ["saya makan nasi enak", "nasi enak saya makan"]:   # peluang kalimat
    print(f"ln p('{kal}') = {np.log(maju_mundur(kal.split())[1]):.2f}")

## Listing 13.3: Filter Kalman dan penghalus RTS untuk pelacakan kapal

Berkas: `kode/bab13/kalman_kapal.py`

In [ ]:
import numpy as np

# Kapal di selat (ilustrasi): keadaan z = [x, y, vx, vy] dalam km dan km/menit.
rng = np.random.default_rng(2)
T, dt = 120, 1.0
A = np.eye(4); A[0, 2] = A[1, 3] = dt                   # kecepatan konstan  # (1)
G = np.eye(4)[:2]                                       # hanya posisi teramati
Q = np.diag([0, 0, 1e-4, 1e-4])                         # percepatan acak kecil
R = 0.4 ** 2 * np.eye(2)                                # derau posisi 0,4 km
z = np.zeros((T, 4)); z[0] = [0, 0, 0.30, 0.05]
for n in range(1, T):
    z[n] = A @ z[n-1] + rng.multivariate_normal(np.zeros(4), Q)
    if n == 78: z[n, 2:] = [0.10, 0.28]                 # berbelok saat sinyal hilang
x = z[:, :2] + rng.normal(0, 0.4, (T, 2))
ada = np.ones(T, bool); ada[70:90] = False              # sinyal hilang 20 menit

def kalman(x, ada):                                     # (2)
    m, P = np.r_[x[0], 0, 0], np.diag([0.16, 0.16, 1, 1])
    M, S, Mp, Sp = [], [], [], []
    for n in range(T):
        if n > 0:                                       # prediksi
            m, P = A @ m, A @ P @ A.T + Q
        Mp.append(m); Sp.append(P)
        if ada[n]:                                      # koreksi dengan pengamatan
            Kg = P @ G.T @ np.linalg.inv(G @ P @ G.T + R)
            m, P = m + Kg @ (x[n] - G @ m), (np.eye(4) - Kg @ G) @ P
        M.append(m); S.append(P)
    return map(np.array, (M, S, Mp, Sp))

def rts(M, S, Mp, Sp):                                  # penghalus  # (3)
    Mh, Sh = M.copy(), S.copy()
    for n in range(T - 2, -1, -1):
        J = S[n] @ A.T @ np.linalg.inv(Sp[n+1])
        Mh[n] = M[n] + J @ (Mh[n+1] - Mp[n+1])
        Sh[n] = S[n] + J @ (Sh[n+1] - Sp[n+1]) @ J.T
    return Mh, Sh

M, S, Mp, Sp = kalman(x, ada); Mh, Sh = rts(M, S, Mp, Sp)
galat = lambda e, m: np.sqrt(np.mean(np.sum((e[m, :2] - z[m, :2]) ** 2, 1)))
for nama, e in [("pengamatan", np.c_[x, x]), ("filter", M), ("penghalus", Mh)]:
    print(f"RMSE {nama:10s}: saat ada sinyal {galat(e, ada):.3f} km"
          + ("" if nama == "pengamatan" else f", saat hilang {galat(e, ~ada):.3f} km"))
sd = lambda S_, n: np.sqrt(S_[n, 0, 0])
print(f"simp. baku posisi-x (filter): menit 69 {sd(S, 69):.2f}, menit 89 {sd(S, 89):.2f},"
      f" menit 90 {sd(S, 90):.2f}")